# H3 · Recetario de inferencia estadística

**Estadística II · Ingeniería Industrial · Universidad Industrial de Santander**  
*Herramienta*

**Propósito de aprendizaje**

- Tener a mano, en un solo lugar, las pruebas e intervalos del curso con código que funciona.
- Decidir qué prueba usar según el objetivo y los supuestos (árbol de decisión al final).
- Ver cada procedimiento aplicado a un caso industrial del repositorio.

**Duración sugerida:** Consulta  
**Datos:** casos IC01, IC04, IC05, IC07 y la base `calidad_textil.csv`

> **¿Qué es la inferencia estadística?**
> Es el proceso de sacar conclusiones sobre una *población* a partir de una *muestra*. En lugar de observar a todos los individuos, tomamos un subconjunto y, con procedimientos formales, decidimos qué podemos afirmar —con cierto nivel de confianza— sobre el conjunto completo.

**Flujo general de un análisis inferencial**

```
Pregunta → Datos → Limpieza/filtrado → Revisar supuestos (forma, varianzas) → Prueba e intervalo → Conclusión en el contexto
```

En todo el notebook se usa **α = 0,05**. Si el p-valor < α se rechaza H₀. Cada sección trae: cuándo usar el procedimiento, las hipótesis, una función reutilizable y un ejemplo con datos reales del repositorio.

## 0. Librerías y datos de ejemplo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.diagnostic import lilliefors
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from pathlib import Path
sns.set_theme(style="whitegrid")
ALFA = 0.05

def leer(nombre):
    """Busca el archivo en la carpeta actual o en datos/ del repositorio (en Colab: súbalo)."""
    for p in [".", "..", "../.."]:
        for c in ["", "datos/casos_industriales/csv", "datos/talleres", "datos/clasicos"]:
            r = Path(p) / c / nombre
            if r.exists():
                return pd.read_csv(r)
    from google.colab import files
    print(f"Suba {nombre}"); return pd.read_csv(next(iter(files.upload())))

def decision(p, alfa=ALFA):
    return f"p-valor = {p:.4f} → " + ("se rechaza H₀" if p < alfa else "no se rechaza H₀")

In [ ]:
productividad = leer("IC04_productividad_linea_ensamble.csv")   # una muestra
proveedores   = leer("IC05_resistencia_proveedores.csv")         # tres grupos
compras       = leer("IC07_precios_compras.csv")                 # dos grupos
despachos     = leer("IC01_despachos_centro_distribucion.csv")   # dos variables categóricas
textil        = leer("calidad_textil.csv")                       # proporción
for n, d in [("productividad", productividad), ("proveedores", proveedores), ("compras", compras), ("despachos", despachos), ("textil", textil)]:
    print(f"{n:14s} {d.shape}  columnas: {list(d.columns)}")

## 1. Filtrar subconjuntos

En inferencia casi siempre trabajamos con **subconjuntos bien definidos** (un grupo contra otro, un periodo, un proveedor). Pandas filtra con condiciones booleanas que se combinan con `&` (y), `|` (o) y `~` (no). Cada condición va entre paréntesis.

| Operador | Ejemplo | Significado |
|---|---|---|
| `==`, `!=` | `df["proveedor"] == "Andes"` | igual / diferente |
| `>`, `<`, `>=`, `<=` | `df["precio"] > 50` | comparación numérica |
| `.isin([...])` | `df["zona"].isin(["Zona A", "Zona B"])` | pertenece a la lista |
| `.str.contains("x", case=False, na=False)` | `df["modelo"].str.contains("ford", case=False, na=False)` | contiene el texto |
| `.isna()`, `.notna()` | `df["peso"].notna()` | faltante / no faltante |

> Después de filtrar, revise el tamaño del subconjunto: con muy pocos datos las pruebas pierden potencia y los supuestos son más difíciles de verificar.

In [ ]:
andes = proveedores.loc[proveedores["proveedor"] == "Andes", "resistencia_traccion_mpa"]
contrato = compras.loc[compras["tipo_compra"] == "Contrato marco", "precio_unitario_miles"]
puntual  = compras.loc[compras["tipo_compra"] == "Compra puntual", "precio_unitario_miles"]
print(len(andes), len(contrato), len(puntual))

## 2. Supuesto de normalidad

| Hipótesis | |
|---|---|
| **H₀** | los datos provienen de una distribución normal |
| **H₁** | los datos no provienen de una distribución normal |

- **Shapiro-Wilk** (`stats.shapiro`): la prueba más potente para n pequeño y mediano.
- **Lilliefors** (`lilliefors`): es Kolmogorov-Smirnov **corregido** para cuando la media y la desviación se estiman con los mismos datos. Usar `stats.kstest` con la media y la desviación muestrales da p-valores demasiado grandes (la prueba se vuelve conservadora y casi nunca rechaza).
- Con n grande (cientos o miles) cualquier prueba detecta desviaciones mínimas que no tienen importancia práctica. Por eso **siempre acompañe la prueba con un gráfico Q-Q** y recuerde que, por el Teorema del Límite Central, las pruebas sobre medias toleran bien la no normalidad cuando n es grande.

In [ ]:
def normalidad(x, nombre="variable"):
    x = pd.Series(x).dropna()
    w, p_sw = stats.shapiro(x)
    d, p_li = lilliefors(x, dist="norm")
    print(f"{nombre} (n = {len(x)})")
    print(f"  Shapiro-Wilk  W = {w:.4f}   {decision(p_sw)}")
    print(f"  Lilliefors    D = {d:.4f}   {decision(p_li)}")
    sm.qqplot(x, line="s", markerfacecolor="#00ADC6", markeredgecolor="#00ADC6", alpha=.6)
    plt.title(f"Q-Q normal: {nombre}"); plt.show()

normalidad(productividad["unidades_producidas"], "unidades producidas")

## 3. Igualdad de varianzas

| Hipótesis | |
|---|---|
| **H₀** | σ₁² = σ₂² (= … = σₖ²) |
| **H₁** | al menos una varianza es diferente |

- **F de Fisher** (dos grupos) y **Bartlett** (k grupos): exigen normalidad; son muy sensibles si no se cumple.
- **Levene** (con la mediana, también llamado Brown-Forsythe): robusta ante la no normalidad. Es la opción por defecto.
- **Fligner-Killeen**: no paramétrica, basada en rangos.

El resultado decide qué versión de la prueba t usar (Student o Welch) y si el ANOVA clásico es adecuado.

In [ ]:
def varianzas(*grupos, nombres=None):
    grupos = [pd.Series(g).dropna() for g in grupos]
    nombres = nombres or [f"g{i+1}" for i in range(len(grupos))]
    print("Varianzas muestrales:", ", ".join(f"{n} = {g.var():.3f}" for n, g in zip(nombres, grupos)))
    if len(grupos) == 2:
        f = grupos[0].var() / grupos[1].var(); gl1, gl2 = len(grupos[0]) - 1, len(grupos[1]) - 1
        p_f = 2 * min(stats.f.cdf(f, gl1, gl2), stats.f.sf(f, gl1, gl2))
        print(f"  F de Fisher     F = {f:.4f}   {decision(p_f)}")
    print(f"  Bartlett        {decision(stats.bartlett(*grupos).pvalue)}")
    print(f"  Levene (mediana){' ':1s}{decision(stats.levene(*grupos, center='median').pvalue)}")
    print(f"  Fligner-Killeen {decision(stats.fligner(*grupos).pvalue)}")

varianzas(contrato, puntual, nombres=["contrato marco", "compra puntual"])

### Intervalo de confianza para la varianza (una muestra normal)

$$\left[\frac{(n-1)s^2}{\chi^2_{1-\alpha/2,\,n-1}}\ ;\ \frac{(n-1)s^2}{\chi^2_{\alpha/2,\,n-1}}\right]$$

Este intervalo **sí** depende fuertemente de la normalidad (no lo protege el TLC).

In [ ]:
def ic_varianza(x, confianza=0.95):
    x = pd.Series(x).dropna(); n, s2 = len(x), x.var()
    a = 1 - confianza
    inf = (n - 1) * s2 / stats.chi2.ppf(1 - a / 2, n - 1)
    sup = (n - 1) * s2 / stats.chi2.ppf(a / 2, n - 1)
    print(f"s² = {s2:.3f} | IC {confianza:.0%} para σ²: [{inf:.3f} ; {sup:.3f}] | para σ: [{inf**.5:.3f} ; {sup**.5:.3f}]")
    return inf, sup

_ = ic_varianza(andes)

## 4. Una media

**Prueba t de una muestra.** Contrasta la media poblacional contra un valor de referencia μ₀.

| | Bilateral | Unilateral derecha | Unilateral izquierda |
|---|---|---|---|
| H₀ | μ = μ₀ | μ ≤ μ₀ | μ ≥ μ₀ |
| H₁ | μ ≠ μ₀ | μ > μ₀ | μ < μ₀ |
| `alternative=` | `"two-sided"` | `"greater"` | `"less"` |

$$IC_{1-\alpha}(\mu) = \bar x \pm t_{\alpha/2,\,n-1}\,\frac{s}{\sqrt n}$$

Si los datos no son normales y n es pequeño, use **Wilcoxon de rangos con signo** (supone simetría alrededor de la mediana) o la **prueba del signo** (no supone nada sobre la forma).

**Ejemplo (IC04):** suponga que la meta de la línea es 100 unidades por operario-jornada. ¿La productividad media es menor que la meta?

In [ ]:
def ic_media(x, confianza=0.95):
    x = pd.Series(x).dropna()
    inf, sup = stats.t.interval(confianza, len(x) - 1, loc=x.mean(), scale=stats.sem(x))
    print(f"media = {x.mean():.3f} | IC {confianza:.0%}: [{inf:.3f} ; {sup:.3f}]")
    return inf, sup

x = productividad["unidades_producidas"]
t = stats.ttest_1samp(x, popmean=100, alternative="less")
print(f"t = {t.statistic:.4f}, gl = {len(x)-1}, {decision(t.pvalue)}")
_ = ic_media(x)
w = stats.wilcoxon(x - 100, alternative="less")
print(f"Wilcoxon (mediana < 100): {decision(w.pvalue)}")

## 5. Dos medias

### 5.1 Muestras independientes

| | |
|---|---|
| **H₀** | μ₁ = μ₂ |
| **H₁** | μ₁ ≠ μ₂ (o unilateral) |

- `equal_var=True` → t de Student (varianzas iguales, se usa la varianza combinada).
- `equal_var=False` → t de **Welch** (no supone varianzas iguales). Es una opción segura por defecto.
- **Mann-Whitney U**: alternativa basada en rangos. Contrasta si un grupo tiende a tener valores mayores que el otro; solo se interpreta como comparación de medianas si las dos distribuciones tienen la misma forma.

**Ejemplo (IC07):** ¿el precio unitario es diferente con contrato marco que con compra puntual?

In [ ]:
def ic_diferencia(x1, x2, confianza=0.95, varianzas_iguales=False):
    a, b = pd.Series(x1).dropna(), pd.Series(x2).dropna()
    n1, n2 = len(a), len(b); dif = a.mean() - b.mean()
    if varianzas_iguales:
        sp2 = ((n1 - 1) * a.var() + (n2 - 1) * b.var()) / (n1 + n2 - 2)
        se, gl = np.sqrt(sp2 * (1 / n1 + 1 / n2)), n1 + n2 - 2
    else:
        v1, v2 = a.var() / n1, b.var() / n2
        se, gl = np.sqrt(v1 + v2), (v1 + v2) ** 2 / (v1 ** 2 / (n1 - 1) + v2 ** 2 / (n2 - 1))
    tc = stats.t.ppf((1 + confianza) / 2, gl)
    print(f"x̄1 − x̄2 = {dif:.3f} | IC {confianza:.0%} ({'Student' if varianzas_iguales else 'Welch'}, gl = {gl:.1f}): [{dif - tc*se:.3f} ; {dif + tc*se:.3f}]")
    return dif - tc * se, dif + tc * se

print(f"Welch:   {decision(stats.ttest_ind(contrato, puntual, equal_var=False).pvalue)}")
print(f"Student: {decision(stats.ttest_ind(contrato, puntual, equal_var=True).pvalue)}")
_ = ic_diferencia(contrato, puntual)
print(f"Mann-Whitney: {decision(stats.mannwhitneyu(contrato, puntual).pvalue)}")

### 5.2 Muestras pareadas (antes/después, mismo individuo)

Cuando cada dato de un grupo está emparejado con uno del otro (la misma máquina antes y después de un ajuste), se trabaja con las **diferencias** dᵢ = xᵢ − yᵢ y se aplica una prueba t de una muestra sobre d con H₀: μ_d = 0. Alternativa no paramétrica: Wilcoxon de rangos con signo.

Los casos del repositorio no tienen datos pareados; el ejemplo usa datos simulados.

In [ ]:
rng = np.random.default_rng(29006)
antes = rng.normal(52, 6, 25)                 # tiempo de ciclo antes del ajuste (s)
despues = antes - rng.normal(2.0, 2.5, 25)    # el mismo puesto después del ajuste
d = antes - despues
print(f"media de las diferencias = {d.mean():.3f} s")
print(f"t pareada (H₁: μ_d > 0): {decision(stats.ttest_rel(antes, despues, alternative='greater').pvalue)}")
_ = ic_media(d)
print(f"Wilcoxon pareado:        {decision(stats.wilcoxon(antes, despues, alternative='greater').pvalue)}")

## 6. Más de dos medias: ANOVA de un factor

| | |
|---|---|
| **H₀** | μ₁ = μ₂ = … = μₖ |
| **H₁** | al menos una media es diferente |

Supuestos: independencia, normalidad de los residuos y varianzas iguales. Si se rechaza H₀, las **comparaciones múltiples de Tukey** indican qué pares difieren. Si las varianzas son distintas use **ANOVA de Welch**; si los datos son muy asimétricos, **Kruskal-Wallis**.

**Ejemplo (IC05):** ¿la resistencia a la tracción media es la misma para los tres proveedores?

In [ ]:
modelo = smf.ols("resistencia_traccion_mpa ~ C(proveedor)", data=proveedores).fit()
display(sm.stats.anova_lm(modelo, typ=2).round(4))
print("Normalidad de residuos:", decision(stats.shapiro(modelo.resid).pvalue))
grupos = [g["resistencia_traccion_mpa"] for _, g in proveedores.groupby("proveedor")]
print("Levene:", decision(stats.levene(*grupos).pvalue))
from statsmodels.stats.oneway import anova_oneway
print("ANOVA de Welch:", decision(anova_oneway(proveedores["resistencia_traccion_mpa"], proveedores["proveedor"], use_var="unequal").pvalue))
print("Kruskal-Wallis:", decision(stats.kruskal(*grupos).pvalue))
print(pairwise_tukeyhsd(proveedores["resistencia_traccion_mpa"], proveedores["proveedor"]))

## 7. Proporciones

### 7.1 Una proporción

H₀: π = π₀. La aproximación normal requiere n·π₀ ≥ 5 y n·(1 − π₀) ≥ 5.

$$IC_{\text{Wald}} = \hat p \pm z_{\alpha/2}\sqrt{\frac{\hat p(1-\hat p)}{n}}$$

El intervalo de Wald puede salirse de [0, 1] y tiene mala cobertura cuando p̂ está cerca de 0 o 1. **Wilson** o **Agresti-Coull** son preferibles.

**Ejemplo (calidad textil):** el contrato exige que al menos el 95 % de las prendas tenga el color dentro de la tolerancia. H₀: π ≥ 0,95 vs. H₁: π < 0,95.

In [ ]:
def ic_proporcion(exitos, n, confianza=0.95):
    for m in ["normal", "wilson", "agresti_coull"]:
        inf, sup = proportion_confint(exitos, n, alpha=1 - confianza, method=m)
        print(f"  IC {confianza:.0%} {('Wald' if m=='normal' else m):14s} [{inf:.4f} ; {sup:.4f}]")

exitos, n = int((textil["color_conforme"] == "Si").sum()), len(textil)
z, p = proportions_ztest(exitos, n, value=0.95, alternative="smaller", prop_var=0.95)
print(f"p̂ = {exitos/n:.4f} ({exitos}/{n}),  z = {z:.3f},  {decision(p)}")
ic_proporcion(exitos, n)

### 7.2 Dos proporciones

H₀: π₁ = π₂. **Ejemplo (IC01):** ¿la proporción de despachos a tiempo es la misma en el turno de la mañana y en el de la noche?

In [ ]:
sub = despachos[despachos["turno_preparacion"].isin(["Mañana", "Noche"])]
tab = pd.crosstab(sub["turno_preparacion"], sub["cumplimiento_despacho"] == "A tiempo")
exitos = tab[True].to_numpy(); n_obs = tab.sum(axis=1).to_numpy()
z, p = proportions_ztest(exitos, n_obs)
print(tab.rename(columns={True: "a tiempo", False: "con retraso"}))
print(f"p̂ = {dict(zip(tab.index, (exitos/n_obs).round(4)))}   z = {z:.3f}, {decision(p)}")

## 8. Pruebas χ²

### 8.1 Bondad de ajuste para frecuencias

H₀: las frecuencias siguen la distribución especificada. Cada frecuencia **esperada** debe ser ≥ 5; si no, se agrupan categorías. Si los parámetros del modelo se estiman con los datos, se resta un grado de libertad por cada parámetro estimado (`ddof`).

**Ejemplo (IC01):** ¿los despachos se reparten por igual entre las cinco zonas del almacén?

In [ ]:
obs = despachos["zona_almacen"].value_counts().sort_index()
res = stats.chisquare(obs)          # sin f_exp → reparto uniforme
print(obs.to_dict())
print(f"χ² = {res.statistic:.3f}, gl = {len(obs)-1}, {decision(res.pvalue)}")

### 8.2 Independencia (tabla de contingencia)

H₀: las dos variables categóricas son independientes. Requiere frecuencias esperadas ≥ 5 (como guía, no más del 20 % de las celdas por debajo de 5 y ninguna por debajo de 1). Para tablas 2×2 pequeñas use la **prueba exacta de Fisher** (`stats.fisher_exact`). La **V de Cramér** mide la fuerza de la asociación (0 = ninguna, 1 = perfecta).

El notebook **E2-04** desarrolla este tema con más detalle.

In [ ]:
tabla = pd.crosstab(despachos["turno_preparacion"], despachos["cumplimiento_despacho"])
chi2, p, gl, esperadas = stats.chi2_contingency(tabla)
v = np.sqrt(chi2 / (tabla.values.sum() * (min(tabla.shape) - 1)))
display(tabla)
print(f"χ² = {chi2:.3f}, gl = {gl}, {decision(p)},  esperada mínima = {esperadas.min():.1f},  V de Cramér = {v:.3f}")

---
## Resumen: ¿qué prueba usar?

```
¿Qué quiero inferir?
│
├── Forma de la distribución
│   ├── ¿Es normal?               → Shapiro-Wilk o Lilliefors + gráfico Q-Q
│   └── ¿Sigue otro modelo?       → χ² de bondad de ajuste / Anderson-Darling (ver H2)
│
├── Varianzas
│   ├── Una varianza              → IC con χ² (exige normalidad)
│   ├── Dos o más, datos normales → F (dos grupos) o Bartlett
│   └── Opción robusta            → Levene (mediana) o Fligner-Killeen
│
├── Una media
│   ├── Normal o n grande         → t de una muestra + ic_media()
│   └── n pequeño y no normal     → Wilcoxon con signo / prueba del signo
│
├── Dos medias
│   ├── Independientes            → t de Welch (o Student si las varianzas son iguales) + ic_diferencia()
│   │   └── no normales, n pequeño → Mann-Whitney U
│   └── Pareadas                  → t pareada (sobre las diferencias) / Wilcoxon pareado
│
├── Tres o más medias             → ANOVA + Tukey  |  ANOVA de Welch  |  Kruskal-Wallis
│
├── Proporciones
│   ├── Una                       → z + IC de Wilson
│   └── Dos                       → z de dos proporciones
│
└── Variables categóricas
    ├── Frecuencias vs. modelo    → χ² de bondad de ajuste
    └── Asociación entre dos      → χ² de independencia (Fisher si 2×2 pequeña) + V de Cramér
```